In [ ]:
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split
import gc

# 1. Mount Drive
drive.mount('/content/drive')

# Path to your file in Drive
file_path = '/content/drive/My Drive/clinical_trials_embeddings/clinical_trials_full_embedded.parquet'

# 2. Load Data (Select only columns you need)
selected_columns = [
    'phases', 'study_type', 'enrollment_count', 'lead_sponsor_class',
    'sex', 'minimum_age', 'maximum_age', 'overall_status',
    'brief_summary_embedding', 'eligibility_criteria_embedding'
]

print("Loading dataset...")
full_df = pd.read_parquet(file_path, columns=selected_columns)

# 3. Create Splits
# Split 1: Test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    full_df,
    full_df['overall_status'],
    test_size=0.15,
    stratify=full_df['overall_status'],
    random_state=42
)

# Free up memory
del full_df
gc.collect()

# Split 2: Train/Val (Split Temp into Train/Val)
# 0.15 / 0.85 = ~0.1765
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    stratify=y_temp,
    random_state=42
)

# Free up memory
del X_temp, y_temp
gc.collect()

# 4. Save to Drive as 3 separate files
save_path = '/content/drive/My Drive/clinical_trials_embeddings/'

print("Saving Train set...")
X_train.to_parquet(save_path + 'train_set.parquet')

print("Saving Val set...")
X_val.to_parquet(save_path + 'val_set.parquet')

print("Saving Test set...")
X_test.to_parquet(save_path + 'test_set.parquet')

print("Done! You can now download these files to your local machine.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset...
Saving Train set...


In [ ]:
# 1. Install Polars (It's not installed by default in Colab yet)
!pip install polars pyarrow

import polars as pl
from sklearn.model_selection import train_test_split
from google.colab import drive
import gc

# 2. Mount Drive
drive.mount('/content/drive')

# UPDATE THIS PATH to your actual file location
input_path = '/content/drive/My Drive/clinical_trials_embeddings/clinical_trials_full_embedded.parquet'
output_dir = '/content/drive/My Drive/clinical_trials_embeddings/'

# --- STEP 1: Load ONLY metadata to calculate the split ---
print("Scanning dataset to get IDs...")

# We use 'scan_parquet' which is Lazy. It doesn't load data yet.
# We explicitly fetch only the target column to calculate stratify.
# We also add a row_index so we know which rows are which.
lazy_df = pl.scan_parquet(input_path).with_row_index(name="row_id")

# Collect ONLY the row_id and the label into memory (This is very small, < 50MB)
# Note: Ensure 'overall_status' is the exact name of your label column
meta_df = lazy_df.select(['row_id', 'overall_status']).collect()

print(f"Total rows found: {meta_df.height}")

# --- STEP 2: Calculate the Split Indices ---
print("Calculating splits...")

# We split the Row IDs, not the actual data
# 1. Split Test (15%)
train_temp_ids, test_ids = train_test_split(
    meta_df['row_id'],
    test_size=0.15,
    stratify=meta_df['overall_status'],
    random_state=42
)

# Get the labels for the temp set to stratify the second split
# We filter meta_df to get the labels matching train_temp_ids
temp_labels = meta_df.filter(pl.col('row_id').is_in(train_temp_ids))['overall_status']

# 2. Split Train/Val (17.65% of remaining = 15% of total)
train_ids, val_ids = train_test_split(
    train_temp_ids,
    test_size=0.1765,
    stratify=temp_labels,
    random_state=42
)

# Clean up memory
del meta_df, temp_labels, train_temp_ids
gc.collect()

print(f"Split sizes determined:")
print(f"Train rows: {len(train_ids)}")
print(f"Val rows:   {len(val_ids)}")
print(f"Test rows:  {len(test_ids)}")

# --- STEP 3: Write the files using Streaming (Low RAM) ---
# This looks at the original big file, picks the rows we want,
# and saves them without ever holding the whole file in RAM.

def save_subset(subset_ids, file_name):
    print(f"Processing and saving {file_name}...")

    # Define the columns you want to keep
    # (If you want ALL columns, remove the select() part)
    columns_to_keep = [
        'phases', 'study_type', 'enrollment_count', 'lead_sponsor_class',
        'sex', 'minimum_age', 'maximum_age', 'overall_status',
        'brief_summary_embedding', 'eligibility_criteria_embedding'
    ]

    # 1. Connect to original file
    # 2. Filter for the specific row IDs
    # 3. Select columns
    # 4. Sink (Write) to parquet
    (
        pl.scan_parquet(input_path)
        .with_row_index(name="row_id")
        .filter(pl.col("row_id").is_in(subset_ids))
        .select(columns_to_keep)
        .sink_parquet(output_dir + file_name)
    )
    print(f"Saved {file_name} successfully.")

# Save Test
save_subset(test_ids, 'test_set.parquet')

# Save Val
save_subset(val_ids, 'val_set.parquet')

# Save Train
# This is the biggest one. Polars handles the memory management automatically.
save_subset(train_ids, 'train_set.parquet')

print("All Done! You have 3 separate files in your Drive now.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scanning dataset to get IDs...
Total rows found: 541897
Calculating splits...


/tmp/ipython-input-3235060325.py:44: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  temp_labels = meta_df.filter(pl.col('row_id').is_in(train_temp_ids))['overall_status']


Split sizes determined:
Train rows: 379313
Val rows:   81299
Test rows:  81285
Processing and saving test_set.parquet...


/tmp/ipython-input-3235060325.py:87: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .sink_parquet(output_dir + file_name)


Saved test_set.parquet successfully.
Processing and saving val_set.parquet...
Saved val_set.parquet successfully.
Processing and saving train_set.parquet...
Saved train_set.parquet successfully.
All Done! You have 3 separate files in your Drive now.
